# Hinglish Grammarly — Colab Model Server

This notebook:
1. Installs dependencies
2. Loads the 4B Hinglish correction model (once)
3. Starts a FastAPI server exposing `POST /correct`
4. Tunnels the server to a public URL via ngrok
5. Registers that URL with our backend so React never needs to know about it
6. Keeps the server running

**Secrets required** (set these via Colab's "Secrets" panel — the key icon in the left sidebar — or as environment variables; never hardcode them in cells):
- `BACKEND_URL` — e.g. `https://your-backend-url.com`
- `BACKEND_REGISTRATION_TOKEN` — must match the backend's `BACKEND_REGISTRATION_TOKEN`
- `NGROK_AUTH_TOKEN` — from https://dashboard.ngrok.com/get-started/your-authtoken
- `MODEL_NAME` (optional) — defaults to `google/gemma-3-4b-it`
- `HF_TOKEN` (optional) — only needed if the model repo is gated

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn[standard] pyngrok nest_asyncio transformers accelerate requests

In [ ]:
# 2. Load secrets into environment variables.
# Prefer Colab's built-in Secrets manager (google.colab.userdata) so nothing
# is ever hardcoded or committed to source control.
import os

try:
    from google.colab import userdata

    def _secret(name, default=None):
        try:
            return userdata.get(name)
        except Exception:
            return os.environ.get(name, default)
except ImportError:
    def _secret(name, default=None):
        return os.environ.get(name, default)

os.environ["BACKEND_URL"] = _secret("BACKEND_URL", "")
os.environ["BACKEND_REGISTRATION_TOKEN"] = _secret("BACKEND_REGISTRATION_TOKEN", "")
os.environ["NGROK_AUTH_TOKEN"] = _secret("NGROK_AUTH_TOKEN", "")
os.environ["MODEL_NAME"] = _secret("MODEL_NAME", "google/gemma-3-4b-it")
hf_token = _secret("HF_TOKEN", "")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

assert os.environ["BACKEND_URL"], "Set BACKEND_URL in Colab Secrets."
assert os.environ["BACKEND_REGISTRATION_TOKEN"], "Set BACKEND_REGISTRATION_TOKEN in Colab Secrets."
assert os.environ["NGROK_AUTH_TOKEN"], "Set NGROK_AUTH_TOKEN in Colab Secrets."
print("Secrets loaded.")

In [ ]:
# 3. Fetch this project's colab/ source files.
# If you uploaded colab_server.py and register.py to the Colab filesystem
# (Files panel, or by mounting Drive), this cell is a no-op check.
import os

for fname in ["colab_server.py", "register.py"]:
    assert os.path.exists(fname), (
        f"{fname} not found. Upload it to the Colab filesystem "
        "(left sidebar -> Files -> upload) before continuing."
    )
print("Source files present.")

In [ ]:
# 4. Load the model and start FastAPI (model loads once, here).
import nest_asyncio
import uvicorn
import threading

from colab_server import app  # noqa: E402  (loads the model on import)

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server starting on port 8000...")

In [ ]:
# 5. Open a public tunnel to the local FastAPI server.
import time
from pyngrok import ngrok, conf

conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]

# Give uvicorn a moment to bind before tunneling.
time.sleep(3)

public_tunnel = ngrok.connect(8000, "http")
public_url = public_tunnel.public_url
print(f"Public URL: {public_url}")

In [ ]:
# 6. Register this URL with our backend, so React/backend always know
#    where the currently active model lives.
from register import register_with_backend

register_with_backend(public_url)

In [ ]:
# 7. Keep this cell running to keep the server + tunnel alive.
# Re-run cells 5 and 6 if the tunnel ever drops (e.g. after Colab idles out),
# since a new tunnel means a new public URL that must be re-registered.
import requests

while True:
    time.sleep(60)
    try:
        r = requests.get(f"{public_url}/health", timeout=5)
        print(f"Health check: {r.status_code} {r.json()}")
    except Exception as exc:
        print(f"Health check failed: {exc}")